In [1]:
# ============================================================
# Cell 1: Imports
# ============================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             brier_score_loss, confusion_matrix, fbeta_score)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import shap
import joblib
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("✅ Libraries imported successfully!")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")

✅ Libraries imported successfully!
PyTorch : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU
Device  : cuda


In [2]:
# ============================================================
# Cell 2: Paths & Constants
# ============================================================

# المسارات
DATA_DIR    = "data"
MODELS_DIR  = "models"
RESULTS_DIR = "results"
PLOTS_DIR   = "plots"

TRAIN_PATH = os.path.join(DATA_DIR, "train_forecast24_undersampled005_patientaware.csv")
VAL_PATH   = os.path.join(DATA_DIR, "val_forecast24.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test_forecast24.csv")

# نفس الـ 13 features المعتمدة — عدالة المقارنة
FEATURE_COLS = [
    'mean_hr', 'mean_spo2', 'sd_hr', 'sd_spo2',
    'skewness_hr', 'skewness_spo2', 'kurtosis_hr', 'kurtosis_spo2',
    'max_xc_hr_spo2', 'min_xc_hr_spo2',
    'sub', 'sepsis_window', 'blackout_window'
]

TARGET_COL  = 'y_forecast_24h'
PATIENT_COL = 'new_id'
TIME_COL    = 'seconds_since_birth'

# Hyperparameters
BATCH_SIZE  = 256
EPOCHS      = 50
LR          = 1e-3

print("✅ Paths & Constants defined!")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

✅ Paths & Constants defined!
Features (13): ['mean_hr', 'mean_spo2', 'sd_hr', 'sd_spo2', 'skewness_hr', 'skewness_spo2', 'kurtosis_hr', 'kurtosis_spo2', 'max_xc_hr_spo2', 'min_xc_hr_spo2', 'sub', 'sepsis_window', 'blackout_window']


In [3]:
# ============================================================
# Cell 3: Load Data
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("✅ Data loaded!")
print(f"Train : {train_df.shape}")
print(f"Val   : {val_df.shape}")
print(f"Test  : {test_df.shape}")

print(f"\n📊 Class Distribution:")
print(f"Train positive: {train_df[TARGET_COL].mean():.4f}")
print(f"Val   positive: {val_df[TARGET_COL].mean():.4f}")
print(f"Test  positive: {test_df[TARGET_COL].mean():.4f}")

✅ Data loaded!
Train : (45717, 19)
Val   : (769979, 18)
Test  : (617005, 18)

📊 Class Distribution:
Train positive: 0.6601
Val   positive: 0.0075
Test  positive: 0.0125


In [4]:
# ============================================================
# Cell 4: Preprocessing & DataLoader
# ============================================================

# Features & Labels
X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET_COL].values
train_patients = train_df[PATIENT_COL].values
train_times    = train_df[TIME_COL].values

X_val = val_df[FEATURE_COLS].values
y_val = val_df[TARGET_COL].values
val_patients = val_df[PATIENT_COL].values
val_times    = val_df[TIME_COL].values

X_test = test_df[FEATURE_COLS].values
y_test = test_df[TARGET_COL].values
test_patients = test_df[PATIENT_COL].values
test_times    = test_df[TIME_COL].values

# Scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(MODELS_DIR, "cnn_scaler.pkl"))

# Class weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(device)

# Dataset
class SepsisDataset(Dataset):
    def __init__(self, X, y):
        # 1D-CNN يحتاج shape: (batch, channels, features)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# DataLoaders
train_loader = DataLoader(SepsisDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(SepsisDataset(X_val, y_val),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(SepsisDataset(X_test, y_test),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print("✅ Preprocessing done!")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"pos_weight: {pos_weight.item():.4f}")
print(f"Sample batch shape: {next(iter(train_loader))[0].shape}")

✅ Preprocessing done!
X_train: (45717, 13) | X_val: (769979, 13) | X_test: (617005, 13)
pos_weight: 0.5149
Sample batch shape: torch.Size([256, 1, 13])


In [5]:
# ============================================================
# Cell 5: 1D-CNN Model Architecture
# ============================================================

class CNNModel(nn.Module):
    def __init__(self, input_size, dropout=0.3):
        super(CNNModel, self).__init__()

        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32,
                      kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64,
                      kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Block 3
        self.block3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128,
                      kernel_size=1),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool1d(1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (batch, 1, 13)
        x = self.block1(x)   # (batch, 32, 13)
        x = self.block2(x)   # (batch, 64, 13)
        x = self.block3(x)   # (batch, 128, 13)
        x = self.gap(x)      # (batch, 128, 1)
        x = x.squeeze(-1)    # (batch, 128)
        x = self.classifier(x)  # (batch, 1)
        return x.squeeze(1)     # (batch,)

# إنشاء النموذج
model = CNNModel(
    input_size = len(FEATURE_COLS),
    dropout    = 0.3
).to(device)

# Complexity
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ 1D-CNN Model created!")
print(f"\n{model}")
print(f"\n📊 Model Complexity:")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Non-trainable        : {total_params - trainable_params:,}")

# تحقق من الـ forward pass
sample = torch.randn(4, 1, len(FEATURE_COLS)).to(device)
output = model(sample)
print(f"\nSample input  : {sample.shape}")
print(f"Sample output : {output.shape}")

✅ 1D-CNN Model created!

CNNModel(
  (block1): Sequential(
    (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (block2): Sequential(
    (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (block3): Sequential(
    (0): Conv1d(64, 128, kernel_size=(1,), stride=(1,))
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (gap): AdaptiveAvgPool1d(output_size=1)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

📊 Model Complexity:
Total parameters     :

In [6]:
# ============================================================
# Cell 6: Optimizer, Loss, Scheduler & Training
# ============================================================

# Loss + Optimizer + Scheduler
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True)

# Early Stopping
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_score = None
        self.stop       = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            print(f"   EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stop = True
        else:
            self.best_score = score
            self.counter    = 0

early_stopping = EarlyStopping(patience=10)

# Train & Evaluate functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs  = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            total_loss += loss.item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y_batch.cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    avg_loss   = total_loss / len(loader)
    roc_auc    = roc_auc_score(all_labels, all_probs)
    pr_auc     = average_precision_score(all_labels, all_probs)
    return avg_loss, roc_auc, pr_auc, all_probs, all_labels

# ── Training ──
print("🚀 Training started!")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'ROC-AUC':<12}{'PR-AUC':<12}{'LR'}")
print("-" * 65)

history         = {'train_loss': [], 'val_loss': [], 'roc_auc': [], 'pr_auc': []}
best_pr_auc     = 0
best_model_path = os.path.join(MODELS_DIR, "cnn_best.pt")
training_start  = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, roc_auc, pr_auc, _, _ = evaluate(model, val_loader, criterion, device)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(pr_auc)
    early_stopping(pr_auc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['roc_auc'].append(roc_auc)
    history['pr_auc'].append(pr_auc)

    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        torch.save(model.state_dict(), best_model_path)
        flag = " ⭐"
    else:
        flag = ""

    print(f"{epoch:<8}{train_loss:<14.4f}{val_loss:<12.4f}"
          f"{roc_auc:<12.4f}{pr_auc:<12.4f}{current_lr:.2e}{flag}")

    if early_stopping.stop:
        print(f"\n⏹ Early stopping at epoch {epoch}")
        break

total_time = time.time() - training_start
print(f"\n✅ Training finished!")
print(f"Best PR-AUC : {best_pr_auc:.4f}")
print(f"Total time  : {total_time/60:.2f} minutes")
print(f"Model saved : {best_model_path}")

🚀 Training started!
Epoch   Train Loss    Val Loss    ROC-AUC     PR-AUC      LR
-----------------------------------------------------------------
1       0.3859        0.4446      0.8195      0.5199      1.00e-03 ⭐
2       0.3270        0.4120      0.8285      0.5296      1.00e-03 ⭐
3       0.3153        0.4884      0.8297      0.5319      1.00e-03 ⭐
4       0.3092        0.4064      0.8320      0.5361      1.00e-03 ⭐
   EarlyStopping: 1/10
5       0.3018        0.4075      0.8343      0.5346      1.00e-03
   EarlyStopping: 2/10
6       0.3007        0.4271      0.8332      0.5354      1.00e-03
7       0.2975        0.3673      0.8375      0.5365      1.00e-03 ⭐
   EarlyStopping: 1/10
8       0.2974        0.3576      0.8372      0.5364      1.00e-03
   EarlyStopping: 2/10
9       0.2949        0.4088      0.8347      0.5354      1.00e-03
   EarlyStopping: 3/10
10      0.2938        0.3805      0.8356      0.5347      1.00e-03
   EarlyStopping: 4/10
11      0.2925        0.4205      0

In [7]:
# ============================================================
# Cell 7: Load Best Model & Evaluate
# ============================================================

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("✅ Best model loaded!")

_, val_roc, val_pr, val_probs, val_labels = evaluate(
    model, val_loader, criterion, device)
_, test_roc, test_pr, test_probs, test_labels = evaluate(
    model, test_loader, criterion, device)

print(f"\n📊 Val  → ROC-AUC: {val_roc:.4f} | PR-AUC: {val_pr:.4f}")
print(f"📊 Test → ROC-AUC: {test_roc:.4f} | PR-AUC: {test_pr:.4f}")

✅ Best model loaded!

📊 Val  → ROC-AUC: 0.8375 | PR-AUC: 0.5365
📊 Test → ROC-AUC: 0.8558 | PR-AUC: 0.5325


In [8]:
# ============================================================
# Cell 8: Calibration & Threshold Selection
# ============================================================

# Platt Scaling على الـ Val
platt = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
platt.fit(val_probs.reshape(-1, 1), val_labels)

val_probs_cal  = platt.predict_proba(val_probs.reshape(-1, 1))[:, 1]
test_probs_cal = platt.predict_proba(test_probs.reshape(-1, 1))[:, 1]

joblib.dump(platt, os.path.join(MODELS_DIR, "cnn_calibration.pkl"))

# Brier قبل وبعد
brier_before = brier_score_loss(val_labels, val_probs)
brier_after  = brier_score_loss(val_labels, val_probs_cal)

print(f"✅ Calibration done!")
print(f"Brier Before : {brier_before:.4f}")
print(f"Brier After  : {brier_after:.4f}")

# Threshold على F2-score
thresholds        = np.arange(0.001, 0.5, 0.001)
best_threshold    = 0.5
best_f2           = 0
f2_scores         = []

for thresh in thresholds:
    preds = (val_probs_cal >= thresh).astype(int)
    f2    = fbeta_score(val_labels, preds, beta=2, zero_division=0)
    f2_scores.append(f2)
    if f2 > best_f2:
        best_f2       = f2
        best_threshold = thresh

print(f"\n✅ Threshold selected!")
print(f"Best Threshold : {best_threshold:.3f}")
print(f"Best F2-score  : {best_f2:.4f}")

# حفظ
cal_params = {
    "coef"      : platt.coef_[0][0],
    "intercept" : platt.intercept_[0],
    "method"    : "Platt Scaling"
}
with open(os.path.join(RESULTS_DIR, "cnn_calibration.json"), "w") as f:
    json.dump(cal_params, f, indent=4)

threshold_info = {
    "best_threshold" : float(best_threshold),
    "best_f2"        : float(best_f2),
    "metric"         : "F2-score",
    "selected_on"    : "validation set"
}
with open(os.path.join(RESULTS_DIR, "cnn_threshold.json"), "w") as f:
    json.dump(threshold_info, f, indent=4)

# Calibration Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, probs, title in zip(
    axes,
    [val_probs, val_probs_cal],
    ["Before Calibration", "After Calibration (Platt)"]):
    fraction_pos, mean_pred = calibration_curve(
        val_labels, probs, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, fraction_pos, 'o-', color='steelblue', label='1D-CNN')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'1D-CNN — {title}')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cnn_calibration.png"), dpi=150)
plt.show()
print("Plot saved ✅")

✅ Calibration done!
Brier Before : 0.1094
Brier After  : 0.0038

✅ Threshold selected!
Best Threshold : 0.388
Best F2-score  : 0.5723
Plot saved ✅


In [9]:
# ============================================================
# Cell 9: Window-level Evaluation
# ============================================================

test_preds_cal = (test_probs_cal >= best_threshold).astype(int)
val_preds_cal  = (val_probs_cal  >= best_threshold).astype(int)

acc       = accuracy_score(test_labels, test_preds_cal)
precision = precision_score(test_labels, test_preds_cal, zero_division=0)
recall    = recall_score(test_labels, test_preds_cal, zero_division=0)
f1        = f1_score(test_labels, test_preds_cal, zero_division=0)
f2        = fbeta_score(test_labels, test_preds_cal, beta=2, zero_division=0)
roc_auc   = roc_auc_score(test_labels, test_probs_cal)
pr_auc    = average_precision_score(test_labels, test_probs_cal)
brier     = brier_score_loss(test_labels, test_probs_cal)
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds_cal).ravel()
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
npv         = tn / (tn + fn) if (tn + fn) > 0 else 0

print("=" * 55)
print("   1D-CNN — Window-level Evaluation (Test)")
print("=" * 55)
print(f"  Accuracy    : {acc:.4f}")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")
print(f"  Specificity : {specificity:.4f}")
print(f"  F1-score    : {f1:.4f}")
print(f"  F2-score    : {f2:.4f}")
print(f"  PPV         : {ppv:.4f}")
print(f"  NPV         : {npv:.4f}")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print(f"  PR-AUC      : {pr_auc:.4f}")
print(f"  Brier Score : {brier:.4f}")
print("=" * 55)
print(f"\n  Confusion Matrix:")
print(f"  TN={tn:,}  FP={fp:,}")
print(f"  FN={fn:,}   TP={tp:,}")

# حفظ
window_metrics = {
    "level"       : "window",
    "threshold"   : float(best_threshold),
    "accuracy"    : acc, "precision" : precision,
    "recall"      : recall, "specificity": specificity,
    "f1"          : f1, "f2"         : f2,
    "ppv"         : ppv, "npv"        : npv,
    "roc_auc"     : roc_auc, "pr_auc" : pr_auc,
    "brier"       : brier,
    "tp": int(tp), "tn": int(tn),
    "fp": int(fp), "fn": int(fn)
}
with open(os.path.join(RESULTS_DIR, "cnn_window_metrics.json"), "w") as f:
    json.dump(window_metrics, f, indent=4)

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
cm = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm, annot=True, fmt=',', cmap='Oranges',
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1'])
plt.title('1D-CNN — Confusion Matrix (Window-level)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cnn_confusion_matrix.png"), dpi=150)
plt.show()
print("\n✅ Window-level metrics saved!")

   1D-CNN — Window-level Evaluation (Test)
  Accuracy    : 0.9937
  Precision   : 0.9877
  Recall      : 0.5004
  Specificity : 0.9999
  F1-score    : 0.6642
  F2-score    : 0.5552
  PPV         : 0.9877
  NPV         : 0.9937
  ROC-AUC     : 0.8558
  PR-AUC      : 0.5325
  Brier Score : 0.0065

  Confusion Matrix:
  TN=609,275  FP=48
  FN=3,838   TP=3,844

✅ Window-level metrics saved!


In [10]:
# ============================================================
# Cell 10: Save Predictions & Patient-level
# ============================================================

# حفظ predictions
val_pred_df = pd.DataFrame({
    'patient_id'         : val_df[PATIENT_COL].values,
    'seconds_since_birth': val_df[TIME_COL].values,
    'y_true'             : val_labels,
    'y_prob'             : val_probs,
    'y_prob_cal'         : val_probs_cal,
    'y_pred'             : val_preds_cal
})

test_pred_df = pd.DataFrame({
    'patient_id'         : test_df[PATIENT_COL].values,
    'seconds_since_birth': test_df[TIME_COL].values,
    'y_true'             : test_labels,
    'y_prob'             : test_probs,
    'y_prob_cal'         : test_probs_cal,
    'y_pred'             : test_preds_cal
})

val_pred_df.to_csv(
    os.path.join(RESULTS_DIR, "cnn_val_predictions_patientwise.csv"), index=False)
test_pred_df.to_csv(
    os.path.join(RESULTS_DIR, "cnn_test_predictions_patientwise.csv"), index=False)

print("✅ Predictions saved!")

# Patient-level
sepsis_detected = 0
sepsis_missed   = 0
false_alarm     = 0

for pid, group in test_pred_df.groupby('patient_id'):
    y_true_p = group['y_true'].max()
    y_pred_p = group['y_pred'].max()

    if y_true_p == 1 and y_pred_p == 1:
        sepsis_detected += 1
    elif y_true_p == 1 and y_pred_p == 0:
        sepsis_missed += 1
    elif y_true_p == 0 and y_pred_p == 1:
        false_alarm += 1

print(f"\n{'='*50}")
print(f"   1D-CNN — Patient-level Summary (Test)")
print(f"{'='*50}")
print(f"  ✅ مرضى Sepsis مكتشفين  : {sepsis_detected}/22")
print(f"  ❌ مرضى Sepsis ضاعوا    : {sepsis_missed}/22")
print(f"  ⚠️  إنذارات كاذبة        : {false_alarm}/44")
print(f"{'='*50}")

# حفظ patient metrics
patient_metrics = {
    "level"          : "patient",
    "total_patients" : 66,
    "sepsis"         : 22,
    "normal"         : 44,
    "detected"       : sepsis_detected,
    "missed"         : sepsis_missed,
    "false_alarm"    : false_alarm,
    "sensitivity"    : sepsis_detected / 22,
    "specificity"    : (44 - false_alarm) / 44
}
with open(os.path.join(RESULTS_DIR, "cnn_patient_metrics_test.json"), "w") as f:
    json.dump(patient_metrics, f, indent=4)

print("\n✅ Patient-level saved!")

✅ Predictions saved!

   1D-CNN — Patient-level Summary (Test)
  ✅ مرضى Sepsis مكتشفين  : 22/22
  ❌ مرضى Sepsis ضاعوا    : 0/22
  ⚠️  إنذارات كاذبة        : 7/44

✅ Patient-level saved!


In [11]:
# ============================================================
# Cell 11: ROC, PR & Training History Plots
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr, tpr, _ = roc_curve(test_labels, test_probs_cal)
axes[0].plot(fpr, tpr, color='darkorange', linewidth=2,
             label=f'1D-CNN (AUC = {test_roc:.4f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='darkorange')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('1D-CNN — ROC Curve (Test Set)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR
precision_curve, recall_curve, _ = precision_recall_curve(test_labels, test_probs_cal)
axes[1].plot(recall_curve, precision_curve, color='steelblue', linewidth=2,
             label=f'1D-CNN (AUC = {test_pr:.4f})')
axes[1].axhline(y=test_labels.mean(), color='gray', linestyle='--',
                label=f'Baseline ({test_labels.mean():.4f})')
axes[1].fill_between(recall_curve, precision_curve, alpha=0.1, color='steelblue')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('1D-CNN — Precision-Recall Curve (Test Set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cnn_roc_pr_curves.png"), dpi=150)
plt.show()

# Training History
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'],
             label='Train', color='darkorange', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],
             label='Val', color='steelblue', linewidth=2)
axes[0].set_title('1D-CNN — Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['roc_auc'],
             color='green', linewidth=2)
axes[1].set_title('1D-CNN — ROC-AUC')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['pr_auc'],
             color='purple', linewidth=2)
axes[2].set_title('1D-CNN — PR-AUC')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cnn_training_history.png"), dpi=150)
plt.show()

print("✅ All plots saved!")

✅ All plots saved!


In [12]:
# ============================================================
# Cell 12: Permutation Feature Importance
# ============================================================

def get_predictions_cnn(model, X, device, batch_size=256):
    model.eval()
    all_probs = []
    dataset   = torch.utils.data.TensorDataset(
        torch.tensor(X, dtype=torch.float32).unsqueeze(1),
        torch.zeros(len(X))
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            logits  = model(X_batch)
            probs   = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
    return np.array(all_probs)

# Baseline
baseline_probs = get_predictions_cnn(model, X_val, device)
baseline_prauc = average_precision_score(y_val, baseline_probs)
print(f"Baseline PR-AUC: {baseline_prauc:.4f}")
print("\n🔄 Computing Permutation Importance...")
print(f"{'Feature':<25} {'PR-AUC':<10} {'Drop':<10} {'Importance'}")
print("-" * 60)

perm_results = []
for i, feature in enumerate(FEATURE_COLS):
    X_val_permuted = X_val.copy()
    np.random.shuffle(X_val_permuted[:, i])

    perm_probs = get_predictions_cnn(model, X_val_permuted, device)
    perm_prauc = average_precision_score(y_val, perm_probs)
    drop       = baseline_prauc - perm_prauc

    perm_results.append({
        'feature'    : feature,
        'prauc'      : perm_prauc,
        'drop'       : drop,
        'importance' : max(drop, 0)
    })
    print(f"{feature:<25} {perm_prauc:<10.4f} {drop:<10.4f} "
          f"{'🔴 مهمة جداً' if drop > 0.05 else '🟡 متوسطة' if drop > 0.01 else '🟢 ضعيفة'}")

perm_df = pd.DataFrame(perm_results).sort_values('importance', ascending=False)

# رسمة
plt.figure(figsize=(10, 6))
colors = ['#d32f2f' if x > 0.05 else '#f57c00' if x > 0.01 else '#388e3c'
          for x in perm_df['importance']]
plt.barh(perm_df['feature'], perm_df['importance'], color=colors)
plt.xlabel('Importance (Drop in PR-AUC)')
plt.title('1D-CNN — Permutation Feature Importance')
plt.axvline(x=0.05, color='red', linestyle='--', alpha=0.5, label='High')
plt.axvline(x=0.01, color='orange', linestyle='--', alpha=0.5, label='Medium')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cnn_permutation_importance.png"), dpi=150)
plt.show()

perm_df.to_csv(os.path.join(RESULTS_DIR, "cnn_permutation_importance.csv"), index=False)
print("\n✅ Permutation importance saved!")

Baseline PR-AUC: 0.5365

🔄 Computing Permutation Importance...
Feature                   PR-AUC     Drop       Importance
------------------------------------------------------------
mean_hr                   0.5349     0.0016     🟢 ضعيفة
mean_spo2                 0.5335     0.0030     🟢 ضعيفة
sd_hr                     0.5331     0.0034     🟢 ضعيفة
sd_spo2                   0.5370     -0.0005    🟢 ضعيفة
skewness_hr               0.5352     0.0013     🟢 ضعيفة
skewness_spo2             0.5344     0.0021     🟢 ضعيفة
kurtosis_hr               0.5357     0.0008     🟢 ضعيفة
kurtosis_spo2             0.5351     0.0014     🟢 ضعيفة
max_xc_hr_spo2            0.5369     -0.0004    🟢 ضعيفة
min_xc_hr_spo2            0.5348     0.0017     🟢 ضعيفة
sub                       0.5361     0.0004     🟢 ضعيفة
sepsis_window             0.0134     0.5231     🔴 مهمة جداً
blackout_window           0.5219     0.0146     🟡 متوسطة

✅ Permutation importance saved!


In [13]:
# ============================================================
# Cell 13: SHAP Analysis
# ============================================================

print("🔄 Computing SHAP values...")

class CNNWrapperTrain(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        return torch.sigmoid(out).unsqueeze(1)

wrapped_model = CNNWrapperTrain(model).to(device)

np.random.seed(SEED)
sample_idx   = np.random.choice(len(X_val), 200, replace=False)
X_val_sample = X_val[sample_idx]
X_background = X_val[:50]

X_bg_tensor     = torch.tensor(X_background, dtype=torch.float32).unsqueeze(1).to(device)
X_sample_tensor = torch.tensor(X_val_sample, dtype=torch.float32).unsqueeze(1).to(device)

explainer   = shap.GradientExplainer(wrapped_model, X_bg_tensor)
shap_values = explainer.shap_values(X_sample_tensor)

if isinstance(shap_values, list):
    shap_vals = shap_values[0]
else:
    shap_vals = shap_values

shap_vals = np.array(shap_vals)
while shap_vals.ndim > 2:
    if shap_vals.shape[1] == 1:
        shap_vals = shap_vals.squeeze(1)
    elif shap_vals.shape[-1] == 1:
        shap_vals = shap_vals.squeeze(-1)
    else:
        break

print(f"SHAP shape: {shap_vals.shape}")

# Summary Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_val_sample,
                  feature_names=FEATURE_COLS, show=False, plot_type="dot")
plt.title("1D-CNN — SHAP Summary Plot")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cnn_shap_summary.png"),
            dpi=150, bbox_inches='tight')
plt.show()

# Bar Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_val_sample,
                  feature_names=FEATURE_COLS, show=False, plot_type="bar")
plt.title("1D-CNN — SHAP Feature Importance")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cnn_shap_bar.png"),
            dpi=150, bbox_inches='tight')
plt.show()

# حفظ
mean_shap = np.abs(shap_vals).mean(axis=0)
shap_df   = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'mean_shap' : mean_shap
}).sort_values('mean_shap', ascending=False)

shap_df.to_csv(os.path.join(RESULTS_DIR, "cnn_shap_values.csv"), index=False)

print("\n📊 SHAP Feature Importance:")
print(shap_df.to_string(index=False))
print("\n✅ SHAP saved!")

🔄 Computing SHAP values...
SHAP shape: (200, 13)

📊 SHAP Feature Importance:
        feature  mean_shap
      mean_spo2   0.116958
          sd_hr   0.071137
    skewness_hr   0.040810
        sd_spo2   0.029376
        mean_hr   0.026256
  skewness_spo2   0.018878
  kurtosis_spo2   0.016851
 max_xc_hr_spo2   0.015800
    kurtosis_hr   0.013584
 min_xc_hr_spo2   0.012006
            sub   0.011555
blackout_window   0.004259
  sepsis_window   0.002407

✅ SHAP saved!


In [14]:
# ============================================================
# Cell 14: Final Summary
# ============================================================

# Inference time
model.eval()
start = time.time()
with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        _ = model(X_batch)
inference_time = time.time() - start

model_size  = os.path.getsize(best_model_path) / (1024 * 1024)
scaler_size = os.path.getsize(os.path.join(MODELS_DIR, "cnn_scaler.pkl")) / 1024

final_summary = {
    "model"        : "1D-CNN",
    "features"     : FEATURE_COLS,
    "architecture" : {
        "blocks"           : "Conv1D(32) → Conv1D(64) → Conv1D(128) → GAP",
        "dropout"          : 0.3,
        "batch_norm"       : True,
        "total_params"     : total_params,
        "trainable_params" : trainable_params
    },
    "training" : {
        "optimizer"        : "AdamW",
        "loss"             : "BCEWithLogitsLoss",
        "scheduler"        : "ReduceLROnPlateau",
        "early_stopping"   : "patience=10 on PR-AUC",
        "epochs_trained"   : len(history['train_loss']),
        "best_val_pr_auc"  : best_pr_auc,
        "training_time_min": round(total_time/60, 2)
    },
    "calibration" : {
        "method"       : "Platt Scaling",
        "brier_before" : float(brier_score_loss(val_labels, val_probs)),
        "brier_after"  : float(brier_score_loss(val_labels, val_probs_cal))
    },
    "threshold" : {
        "value"   : float(best_threshold),
        "metric"  : "F2-score",
        "f2_value": float(best_f2)
    },
    "window_metrics" : {
        "accuracy"    : acc, "precision"   : precision,
        "recall"      : recall, "specificity" : specificity,
        "f1"          : f1, "f2"           : f2,
        "ppv"         : ppv, "npv"          : npv,
        "roc_auc"     : roc_auc, "pr_auc"  : pr_auc,
        "brier"       : brier,
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn)
    },
    "patient_metrics" : {
        "total"       : 66, "sepsis"    : 22,
        "normal"      : 44, "detected"  : sepsis_detected,
        "missed"      : sepsis_missed, "false_alarm": false_alarm
    },
    "complexity" : {
        "total_params"    : total_params,
        "training_min"    : round(total_time/60, 2),
        "inference_sec"   : round(inference_time, 2),
        "per_sample_ms"   : round(inference_time/len(test_pred_df)*1000, 4),
        "model_size_mb"   : round(model_size, 2),
        "scaler_size_kb"  : round(scaler_size, 2)
    }
}

with open(os.path.join(RESULTS_DIR, "cnn_final_summary.json"), "w") as f:
    json.dump(final_summary, f, indent=4)

print("=" * 57)
print("   1D-CNN — FINAL COMPLETE SUMMARY")
print("=" * 57)
print(f"\n🧠 Architecture:")
print(f"  Parameters      : {total_params:,}")
print(f"  Blocks          : Conv1D(32)→Conv1D(64)→Conv1D(128)→GAP")
print(f"\n📊 Window-level (Test):")
print(f"  ROC-AUC         : {roc_auc:.4f}")
print(f"  PR-AUC          : {pr_auc:.4f}")
print(f"  Recall          : {recall:.4f}")
print(f"  Precision       : {precision:.4f}")
print(f"  F2-score        : {f2:.4f}")
print(f"  FP (إنذار كاذب) : {fp:,}")
print(f"\n👥 Patient-level (Test):")
print(f"  مكتشفين         : {sepsis_detected}/22 ✅")
print(f"  ضاعوا           : {sepsis_missed}/22")
print(f"  إنذارات كاذبة   : {false_alarm}/44")
print(f"\n⏱️  Complexity:")
print(f"  Training        : {total_time/60:.2f} min")
print(f"  Inference/sample: {inference_time/len(test_pred_df)*1000:.4f} ms")
print(f"  Model size      : {model_size:.2f} MB")
print("=" * 57)
print("\n🎉 1D-CNN Model Complete!")

   1D-CNN — FINAL COMPLETE SUMMARY

🧠 Architecture:
  Parameters      : 23,425
  Blocks          : Conv1D(32)→Conv1D(64)→Conv1D(128)→GAP

📊 Window-level (Test):
  ROC-AUC         : 0.8558
  PR-AUC          : 0.5325
  Recall          : 0.5004
  Precision       : 0.9877
  F2-score        : 0.5552
  FP (إنذار كاذب) : 48

👥 Patient-level (Test):
  مكتشفين         : 22/22 ✅
  ضاعوا           : 0/22
  إنذارات كاذبة   : 7/44

⏱️  Complexity:
  Training        : 12.17 min
  Inference/sample: 0.0445 ms
  Model size      : 0.10 MB

🎉 1D-CNN Model Complete!
